# 📘 CIFAR-10 Image Classification — Week 4 Assignment
## Build an image classification model on CIFAR-10.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("TensorFlow version:", tf.__version__)

# 📥 Step 1: Load Dataset
CIFAR-10 contains **60,000 color images of size 32×32×3** (RGB channels).
- **50,000 training images** — used to learn model weights
- **10,000 test images** — held out to evaluate final performance

Each image belongs to one of 10 mutually exclusive classes.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("Train shape:", x_train.shape)  # (50000, 32, 32, 3)
print("Test shape:", x_test.shape)    # (10000, 32, 32, 3)
print("Train labels shape:", y_train.shape)
print("Pixel value range: [{}, {}]".format(x_train.min(), x_train.max()))

## 🖼️ Visualize Sample Images

In [ ]:
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]], fontsize=10)
    plt.axis("off")
plt.suptitle("Sample CIFAR-10 Images", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 🧹 Step 2: Preprocessing — Normalize Pixel Values

Raw pixel values range from **0 to 255**. We divide by 255.0 to rescale them to **0.0 – 1.0**.

**Why normalize?**
- Gradient updates become more stable (no exploding gradients from large input magnitudes)
- Faster convergence during training
- Keeps all input features on the same scale, which Adam optimizer handles more efficiently

For the ANN, we additionally **flatten** each 32×32×3 image into a 3072-dimensional vector, because Dense layers expect 1D input.

In [ ]:
# Normalize pixel values 0-255 → 0.0-1.0
x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

# Flatten for ANN: (50000, 32, 32, 3) → (50000, 3072)
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)

print("Normalized range: [{:.1f}, {:.1f}]".format(x_train_norm.min(), x_train_norm.max()))
print("ANN input shape (flattened):", x_train_flat.shape)
print("CNN input shape (spatial):", x_train_norm.shape)

# 🔹 Part 1: ANN Model (Baseline)

**Architecture explanation:**
- **Input:** 3072-dimensional flat vector (32×32×3 = 3072 pixels)
- **Dense(512, relu):** First hidden layer with 512 neurons. ReLU activation introduces non-linearity, allowing the network to learn complex patterns.
- **Dropout(0.3):** Randomly deactivates 30% of neurons per batch during training. This prevents overfitting by forcing the network to learn redundant representations.
- **Dense(256, relu):** Second hidden layer further compresses features.
- **Dense(10, softmax):** Output layer. Softmax converts raw logits into a probability distribution across 10 classes. The predicted class is the one with the highest probability.

**Limitation:** By flattening the image, we lose all spatial information — the model has no idea which pixels are neighbors, making it hard to learn shapes or textures.

In [ ]:
ann_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='ANN_Baseline')

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test, verbose=0)
print(f"ANN Test Loss:     {ann_test_loss:.4f}")
print(f"ANN Test Accuracy: {ann_test_acc:.4f} ({ann_test_acc*100:.2f}%)")

# 🔹 Part 2: CNN Model

**Architecture explanation:**

| Layer | Purpose |
|---|---|
| Conv2D(32, 3×3) | Learns 32 low-level feature maps (edges, colors) |
| BatchNormalization | Normalizes activations per batch → faster, stabler training |
| MaxPooling2D(2×2) | Halves spatial dimensions → reduces parameters, adds translation invariance |
| Conv2D(64, 3×3) | Learns 64 mid-level features (textures, corners) |
| BatchNormalization | Same benefit at deeper layer |
| MaxPooling2D(2×2) | Further spatial compression |
| Conv2D(128, 3×3) | Learns 128 high-level features (object parts) |
| Flatten | Converts 3D feature maps to 1D for classification |
| Dense(128, relu) | Combines all features for final classification |
| Dropout(0.4) | Prevents overfitting in dense layers |
| Dense(10, softmax) | Final class probabilities |

**Key advantages over ANN:**
- **Parameter sharing:** A single 3×3 filter is reused across the whole image — far fewer parameters than fully connected
- **Local connectivity:** Each neuron only looks at a small patch, preserving spatial relationships
- **Hierarchical features:** Early layers detect simple patterns; deeper layers combine them into complex object representations

In [ ]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN_Baseline')

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f"CNN Test Loss:     {cnn_test_loss:.4f}")
print(f"CNN Test Accuracy: {cnn_test_acc:.4f} ({cnn_test_acc*100:.2f}%)")

## 📈 Step 6: Compare Validation Accuracy Curves (ANN vs CNN)

**What to observe in the chart:**
- The CNN validation accuracy curve should consistently sit higher than ANN across all epochs
- ANN may plateau early (around 50-55%) due to its inability to capture spatial features
- CNN continues to improve and typically reaches 70-75% at epoch 10
- A large gap between training and validation accuracy indicates overfitting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Validation Accuracy comparison
axes[0].plot(ann_history.history['val_accuracy'], label='ANN Val Acc', color='coral', linewidth=2, marker='o')
axes[0].plot(cnn_history.history['val_accuracy'], label='CNN Val Acc', color='steelblue', linewidth=2, marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('ANN vs CNN — Validation Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation Loss comparison
axes[1].plot(ann_history.history['val_loss'], label='ANN Val Loss', color='coral', linewidth=2, marker='o')
axes[1].plot(cnn_history.history['val_loss'], label='CNN Val Loss', color='steelblue', linewidth=2, marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('ANN vs CNN — Validation Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training Strategy Comparison: ANN vs CNN on CIFAR-10', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 🚀 Step 7: Training Strategy Upgrade — Data Augmentation

**What is Data Augmentation?**
Data augmentation artificially expands the training dataset by applying random transformations to existing images. The model sees slightly different versions of each image each epoch, which:
- Prevents overfitting (model can't memorize exact training images)
- Improves generalization to unseen real-world images
- Acts as a regularization technique at the data level

**Transformations used:**
- **RandomFlip (horizontal):** Mirrors the image left-to-right. A car flipped horizontally is still a car.
- **RandomRotation(0.1):** Rotates up to ±10% of 360° = ±36°. Helps with slightly tilted objects.
- **RandomZoom(0.1):** Zooms in/out by up to 10%. Handles objects at different distances.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
], name='augmentation_pipeline')

aug_cnn_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, 3, activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN_Augmented')

aug_cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

aug_cnn_model.summary()

# Training the augmented model (Task 5 requirement)
aug_history = aug_cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

aug_test_loss, aug_test_acc = aug_cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f"Augmented CNN Test Accuracy: {aug_test_acc:.4f} ({aug_test_acc*100:.2f}%)")

# 📊 Final Comparison Table

In [ ]:
comparison = pd.DataFrame({
    "Model": ["ANN (Baseline)", "CNN (Baseline)", "CNN (Augmented)"],
    "Architecture": ["Dense only", "Conv2D + BN + Pool", "Augment + Conv2D + Pool"],
    "Input Format": ["Flat vector (3072,)", "Spatial (32,32,3)", "Spatial (32,32,3)"],
    "Test Accuracy": [ann_test_acc, cnn_test_acc, aug_test_acc],
    "Test Accuracy %": [f"{ann_test_acc*100:.2f}%", f"{cnn_test_acc*100:.2f}%", f"{aug_test_acc*100:.2f}%"]
})

print(comparison.to_string(index=False))
comparison

---
# 🎓 Student Learning Tasks — All 5 Implemented

The following cells implement all 5 beginner tasks from the assignment.

## ✅ Task 1: Increase ANN Layers and Observe Performance

**What we're testing:** Does adding more Dense layers improve ANN accuracy on images?

**Expected finding:** Adding layers helps marginally, but the fundamental bottleneck remains — flattening destroys spatial information that no number of Dense layers can recover.

In [ ]:
# Task 1: Deeper ANN with more layers
ann_deep_model = models.Sequential([
    layers.Dense(1024, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='ANN_Deep')

ann_deep_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_deep_model.summary()

ann_deep_history = ann_deep_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

_, ann_deep_acc = ann_deep_model.evaluate(x_test_flat, y_test, verbose=0)
print(f"\nBaseline ANN Accuracy: {ann_test_acc*100:.2f}%")
print(f"Deep ANN Accuracy:     {ann_deep_acc*100:.2f}%")
print(f"Improvement: {(ann_deep_acc - ann_test_acc)*100:+.2f}% — marginal gains because spatial info is lost regardless")

## ✅ Task 2: Change CNN Filters from 32 → 64 → 128

**What we're testing:** The original CNN already uses 32→64→128 filter progression. Here we scale it further to 64→128→256 to see if more filters improve accuracy.

**Why filter count matters:** More filters = more distinct feature detectors at each layer. Doubling filters allows the network to learn twice as many visual patterns per layer, at the cost of more computation.

In [ ]:
# Task 2: Scaled-up CNN with larger filter counts (64→128→256)
cnn_wide_model = models.Sequential([
    layers.Conv2D(64, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN_Wide_Filters')

cnn_wide_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_wide_model.summary()

cnn_wide_history = cnn_wide_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

_, cnn_wide_acc = cnn_wide_model.evaluate(x_test_norm, y_test, verbose=0)
print(f"\nBaseline CNN (32→64→128) Accuracy: {cnn_test_acc*100:.2f}%")
print(f"Wide CNN    (64→128→256) Accuracy: {cnn_wide_acc*100:.2f}%")

## ✅ Task 3: Increase Epochs to 20

**What we're testing:** Training the CNN for 20 epochs instead of 10 to see if the model continues to improve.

**What to watch for:**
- If val_accuracy keeps rising: model still has learning capacity
- If val_accuracy plateaus while train_accuracy rises: overfitting is occurring
- The gap between train and val accuracy is the overfitting gap

In [ ]:
# Task 3: Train CNN for 20 epochs
cnn_20ep_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN_20_Epochs')

cnn_20ep_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_20ep_history = cnn_20ep_model.fit(
    x_train_norm, y_train,
    epochs=20,              # <-- increased to 20
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

_, cnn_20ep_acc = cnn_20ep_model.evaluate(x_test_norm, y_test, verbose=0)
print(f"\nCNN @ 10 Epochs: {cnn_test_acc*100:.2f}%")
print(f"CNN @ 20 Epochs: {cnn_20ep_acc*100:.2f}%")

# Plot 20 epoch curves
plt.figure(figsize=(10, 4))
plt.plot(cnn_20ep_history.history['accuracy'], label='Train Acc', color='steelblue')
plt.plot(cnn_20ep_history.history['val_accuracy'], label='Val Acc', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('CNN Training — 20 Epochs (Train vs Validation Accuracy)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## ✅ Task 4: Add EarlyStopping

**What is EarlyStopping?**
EarlyStopping is a Keras callback that monitors a metric (usually `val_loss`) and automatically stops training when the metric stops improving for a specified number of epochs (`patience`).

**Why use it?**
- Saves compute time — no need to guess the optimal epoch count
- Prevents overfitting — stops before the model memorizes training data
- `restore_best_weights=True` rolls back to the epoch with the best val_loss, not the last one

**Parameters:**
- `monitor='val_loss'`: Watch validation loss
- `patience=5`: Stop if no improvement for 5 consecutive epochs
- `restore_best_weights=True`: Use weights from the best epoch

In [ ]:
# Task 4: CNN with EarlyStopping callback
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

cnn_es_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN_EarlyStopping')

cnn_es_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_es_history = cnn_es_model.fit(
    x_train_norm, y_train,
    epochs=50,               # Set high; EarlyStopping will stop it
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop],  # <-- EarlyStopping added here
    verbose=1
)

_, cnn_es_acc = cnn_es_model.evaluate(x_test_norm, y_test, verbose=0)
actual_epochs = len(cnn_es_history.history['val_loss'])
print(f"\nTraining stopped at epoch: {actual_epochs} (out of 50 max)")
print(f"CNN with EarlyStopping Test Accuracy: {cnn_es_acc*100:.2f}%")

## ✅ Task 5: Data Augmentation Training Run

The augmented model was already built and trained above in Step 7. This cell visualizes the augmentation effect and runs a comprehensive final comparison.

**Augmentation visualization:** We show what the same image looks like after augmentation — slight flips, rotations, and zooms. Each epoch, the model sees augmented variants, making it robust to these transformations.

In [ ]:
# Task 5: Visualize augmentation effect + compare aug vs non-aug

# Show augmented versions of a sample image
sample_img = x_train_norm[5:6]  # Shape: (1, 32, 32, 3)

plt.figure(figsize=(12, 4))
plt.subplot(1, 6, 1)
plt.imshow(x_train_norm[5])
plt.title("Original", fontsize=9)
plt.axis('off')

for i in range(5):
    augmented = data_augmentation(sample_img, training=True)
    plt.subplot(1, 6, i+2)
    plt.imshow(augmented[0].numpy())
    plt.title(f"Aug #{i+1}", fontsize=9)
    plt.axis('off')

plt.suptitle('Data Augmentation: Same Image, Different Transformations', fontsize=11)
plt.tight_layout()
plt.show()

# Compare augmented vs non-augmented training curves
plt.figure(figsize=(10, 4))
plt.plot(cnn_history.history['val_accuracy'], label='CNN (no augmentation)', color='steelblue', linewidth=2)
plt.plot(aug_history.history['val_accuracy'], label='CNN (with augmentation)', color='green', linewidth=2, linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.title('Effect of Data Augmentation on Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Observation: Augmentation may show slightly lower early accuracy")
print("but leads to better generalization with more training (less overfitting).")

# 📊 Full Summary: All Models Compared

In [ ]:
# Evaluate all task models
_, ann_deep_acc = ann_deep_model.evaluate(x_test_flat, y_test, verbose=0)
_, cnn_wide_acc = cnn_wide_model.evaluate(x_test_norm, y_test, verbose=0)
_, cnn_20ep_acc = cnn_20ep_model.evaluate(x_test_norm, y_test, verbose=0)
_, cnn_es_acc = cnn_es_model.evaluate(x_test_norm, y_test, verbose=0)
_, aug_test_acc = aug_cnn_model.evaluate(x_test_norm, y_test, verbose=0)

full_comparison = pd.DataFrame({
    "Model": [
        "ANN Baseline (10 epochs)",
        "ANN Deep — Task 1",
        "CNN Baseline (32→64→128, 10 ep)",
        "CNN Wide Filters (64→128→256) — Task 2",
        "CNN 20 Epochs — Task 3",
        "CNN + EarlyStopping — Task 4",
        "CNN + Augmentation — Task 5"
    ],
    "Test Accuracy": [
        ann_test_acc, ann_deep_acc,
        cnn_test_acc, cnn_wide_acc, cnn_20ep_acc, cnn_es_acc, aug_test_acc
    ]
})

full_comparison['Test Accuracy %'] = full_comparison['Test Accuracy'].apply(lambda x: f"{x*100:.2f}%")
full_comparison = full_comparison.sort_values('Test Accuracy', ascending=False).reset_index(drop=True)
print(full_comparison[['Model','Test Accuracy %']].to_string(index=False))
full_comparison

# ✅ Conclusion

### Key Takeaways from this Project:

1. **ANN vs CNN:** The CNN significantly outperforms the ANN on CIFAR-10. ANN treats images as flat vectors and loses all spatial information, while CNN's convolutional filters capture local patterns (edges, textures, shapes) that are critical for image recognition.

2. **Why spatial structure matters:** A 32×32 image has pixels arranged in a 2D grid where neighboring pixels are correlated. ANN destroys this by flattening. CNN exploits it with local receptive fields.

3. **Batch Normalization:** Normalizing activations between layers stabilizes training, allows higher learning rates, and reduces sensitivity to weight initialization. It acts as a regularizer, sometimes reducing the need for Dropout.

4. **Dropout:** Randomly zeroing neurons during training prevents co-adaptation — neurons can't rely on specific other neurons always being present. This forces learning of robust, distributed representations.

5. **Data Augmentation:** The most effective regularization for image models. By showing the model randomly transformed images each epoch, we dramatically increase the effective dataset size and the model's robustness to real-world variation.

6. **EarlyStopping:** A practical training strategy that eliminates the need to manually tune epoch count. By monitoring validation loss with a patience window, it stops at the optimal point and restores the best weights.

7. **Filter scaling (32→64→128):** Increasing filter count at deeper layers follows the principle that deeper features are more abstract and require more channels to represent the expanded feature space after spatial compression via pooling.

### Real-world Implication:
This CIFAR-10 experiment is the foundational stepping stone to understanding state-of-the-art architectures like ResNet, EfficientNet, and Vision Transformers — all of which build on the same core intuition of hierarchical spatial feature extraction.